# 🚀 Hope ML: Production Training Pipeline

This notebook provides a professional, reproducible environment for training the **Canonical Causal Transformer** model. It includes real-time monitoring via TensorBoard, automated Google Drive persistence, and secure model signing.

---

In [ ]:
# @title ## 1. Hardware & Environment Setup
import torch
import sys
import os

# Check GPU availability
gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# Install core dependencies
!pip install -q torch==2.11.0 pandas==3.0.2 numpy==2.4.4 scikit-learn==1.8.0 tqdm cryptography onnx onnxruntime
print("Dependencies installed.")

## 2. Configuration & Secrets

### 🔐 Secure Credentials Setup
To secure your environment, set the following secrets in the **Google Colab "Secrets" tab** (key icon in the left sidebar):

| Secret Name | Description | Required |
| :--- | :--- | :---: |
| `DERIV_DEMO_TOKEN` | API token for your Deriv Demo account | Yes |
| `DERIV_REAL_TOKEN` | API token for your Deriv Real account | Optional |
| `MODEL_SIGNING_KEY` | Hex-encoded Ed25519 private key for model security | Yes |

**Important**: Enable the **"Notebook access"** toggle for these secrets in the Colab UI.

In [ ]:
from google.colab import drive, userdata
import os
import time
import subprocess

# 1. Mount Google Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

REPO_URL = "https://github.com/planetazul3/hope.git"
TARGET_DIR = "/content/drive/MyDrive/hope"

def run_with_backoff(cmd, max_retries=5):
    for i in range(max_retries):
        try:
            subprocess.check_call(cmd, shell=True)
            return True
        except subprocess.CalledProcessError:
            delay = 2 ** i
            print(f"Command failed. Retrying in {delay}s...")
            time.sleep(delay)
    return False

if not os.path.exists(TARGET_DIR):
    print(f"Cloning {REPO_URL} into Google Drive...")
    # Note: Using depth=1 for faster clone in GDrive
    success = run_with_backoff(f"git clone --depth 1 {REPO_URL} {TARGET_DIR}")
else:
    print(f"Pulling latest changes in {TARGET_DIR}...")
    success = run_with_backoff(f"git -C {TARGET_DIR} pull")

if success:
    print("Repository ready in Google Drive.")
    import sys
    scripts_path = os.path.join(TARGET_DIR, "scripts")
    if scripts_path not in sys.path:
        sys.path.append(scripts_path)
else:
    print("Failed to prepare repository.")

# Export secrets to environment variables
try:
    os.environ["MODEL_SIGNING_KEY"] = userdata.get("MODEL_SIGNING_KEY")
    os.environ["DERIV_DEMO_TOKEN"] = userdata.get("DERIV_DEMO_TOKEN")
    os.environ["DERIV_REAL_TOKEN"] = userdata.get("DERIV_REAL_TOKEN")
    os.environ["DERIV_APP_ID"] = "1089" # Default
    print("Secrets loaded into environment.")
except userdata.SecretNotFoundError:
    print("Warning: Secrets not found in Colab userdata. Ensure they are set in the Secrets tab.")


In [ ]:
# @title ## 3.1 Data Distribution Analysis
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "/content/drive/MyDrive/hope/data/ticks.csv"
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, header=None, nrows=10000)
    if df.shape[1] >= 3:
        prices = df.iloc[:, 2]
    else:
        prices = df.iloc[:, 1]
    
    plt.figure(figsize=(10, 4))
    plt.plot(prices)
    plt.title(f"Price Sample from {os.path.basename(CSV_PATH)}")
    plt.xlabel("Ticks")
    plt.ylabel("Price")
    plt.grid(True)
    plt.show()
else:
    print(f"Ticks file not found at {CSV_PATH}. Please run data collection first.")

In [ ]:
# @title ## 4. Training Hyperparameters
BATCH_SIZE = 128 # @param {type:"integer"}
LEARNING_RATE = 0.0001 # @param {type:"number"}
EPOCHS = 100 # @param {type:"integer"}
TRANSFORMER_SEQUENCE_LENGTH = 32 # @param {type:"integer"}
USE_PRETRAINING = True # @param {type:"boolean"}

os.environ["TRANSFORMER_SEQUENCE_LENGTH"] = str(TRANSFORMER_SEQUENCE_LENGTH)
print(f"Configured training for {EPOCHS} epochs with sequence length {TRANSFORMER_SEQUENCE_LENGTH}")

In [ ]:
# @title ## 5. Training & Monitoring
%load_ext tensorboard
import os

# Log directory in GDrive for persistence
LOG_DIR = "/content/drive/MyDrive/hope/logs"
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Launching TensorBoard (logging to {LOG_DIR})...")
%tensorboard --logdir {LOG_DIR}

import train_fixed
train_fixed.main(log_dir=LOG_DIR)